In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import copy


In [3]:
# i wont to test the pooling 



image = torch.randn(3, 16, 32,32)

pooling = nn.MaxPool2d(kernel_size = 2,
                     stride = 2,
                     padding = 0)
print('end=', pooling(image).shape)


# pooling does not change the number of channels, 
# it only changes the spatial dimensions (height and width) of the input image.nnels (3) but the height and width will be halved.

end= torch.Size([3, 16, 16, 16])


In [4]:
image = torch.tensor([
    [1.,2.,3.],
    [4.,5.,6.],
    [7.,8.,9.]
])
print("Image shape:", image.shape)
flat = image.view(-1)
print("Flattened shape:", flat.shape)


Image shape: torch.Size([3, 3])
Flattened shape: torch.Size([9])


In [5]:

import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

# Convert images to tensors and normalize them
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

# Download CIFAR-10
full_train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# Split training data into train and validation
train_dataset, val_dataset = random_split(
    full_train_dataset,
    [45000, 5000],
    generator=torch.Generator().manual_seed(42)
)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Image values:", images.min().item(), images.max().item())


Train samples: 45000
Validation samples: 5000
Test samples: 10000
Images shape: torch.Size([128, 3, 32, 32])
Labels shape: torch.Size([128])
Image values: -1.0 1.0


In [6]:
# Input → Conv2d → ReLU → MaxPool2d → Conv2d → ReLU → MaxPool2d → Flatten → Linear
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature = nn.Sequential(   # layer 1
            nn.Conv2d(in_channels= 3, out_channels=16, kernel_size=3, stride=1 , padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # layer 2
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1 , padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
            
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=32 * 8 * 8 , out_features=10)
        )
        
        
    def forward(self, x):
        x = self.feature(x)
        return  self.classifier(x)
    
    


In [7]:


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = SimpleCNN().to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss()

num_epochs = 20
patience = 3

best_val_loss = float("inf")
best_model_state = None
epochs_without_improvement = 0

start_time = time.time()

for epoch in range(num_epochs):

    # -------------------------
    # 1. Training
    # -------------------------
    model.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        train_loss_sum += loss.item() * batch_size
        train_correct += (
            logits.argmax(dim=1) == labels
        ).sum().item()
        train_total += batch_size

    train_loss = train_loss_sum / train_total
    train_acc = 100 * train_correct / train_total

    # -------------------------
    # 2. Validation
    # -------------------------
    model.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            batch_size = labels.size(0)

            val_loss_sum += loss.item() * batch_size
            val_correct += (
                logits.argmax(dim=1) == labels
            ).sum().item()
            val_total += batch_size

    val_loss = val_loss_sum / val_total
    val_acc = 100 * val_correct / val_total

    # -------------------------
    # 3. Save best checkpoint
    # -------------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        torch.save(
            best_model_state,
            "best_simple_cnn.pth"
        )

        epochs_without_improvement = 0
        status = "Best model saved"

    else:
        epochs_without_improvement += 1
        status = (
            f"No improvement: "
            f"{epochs_without_improvement}/{patience}"
        )

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}% | "
        f"{status}"
    )

    # -------------------------
    # 4. Early stopping
    # -------------------------
    if epochs_without_improvement >= patience:
        print("Early stopping triggered.")
        break

# Restore the best weights
if best_model_state is not None:
    model.load_state_dict(best_model_state)

elapsed_time = time.time() - start_time

print(f"Best Validation Loss: {best_val_loss:.4f}")
print(f"Training Time: {elapsed_time:.2f} seconds")


Epoch [1/20] Train Loss: 1.5738 | Train Acc: 44.17% | Val Loss: 1.3518 | Val Acc: 51.74% | Best model saved
Epoch [2/20] Train Loss: 1.2548 | Train Acc: 56.02% | Val Loss: 1.2114 | Val Acc: 57.44% | Best model saved
Epoch [3/20] Train Loss: 1.1211 | Train Acc: 61.08% | Val Loss: 1.1359 | Val Acc: 60.18% | Best model saved
Epoch [4/20] Train Loss: 1.0335 | Train Acc: 64.22% | Val Loss: 1.0743 | Val Acc: 63.22% | Best model saved
Epoch [5/20] Train Loss: 0.9729 | Train Acc: 66.08% | Val Loss: 1.0353 | Val Acc: 63.92% | Best model saved
Epoch [6/20] Train Loss: 0.9254 | Train Acc: 68.06% | Val Loss: 1.0037 | Val Acc: 65.30% | Best model saved
Epoch [7/20] Train Loss: 0.8887 | Train Acc: 69.28% | Val Loss: 0.9664 | Val Acc: 66.88% | Best model saved
Epoch [8/20] Train Loss: 0.8599 | Train Acc: 70.42% | Val Loss: 0.9620 | Val Acc: 66.44% | Best model saved
Epoch [9/20] Train Loss: 0.8360 | Train Acc: 71.10% | Val Loss: 0.9562 | Val Acc: 66.96% | Best model saved
Epoch [10/20] Train Loss: 0.